# Tech Challenge Fase 2 — Validações iniciais

Este notebook documenta o **saneamento inicial** antes da EDA e da modelagem.
O objetivo é verificar se o CSV tem estrutura, tipos, identificadores, rótulos
e distribuição compatíveis com o desafio.

**Princípio:** validar e tornar explícitas as decisões; não apagar observações
sem evidência de erro de medição ou justificativa de negócio.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = next(
    (
        candidate
        for candidate in [Path.cwd(), *Path.cwd().parents]
        if (candidate / "data" / "raw" / "WineQT.csv").exists()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError(
        "Execute o notebook a partir do projeto Tech-Challenge-Fase-2-"
        "Classificacao-de-Vinhos ou de uma pasta filha."
    )

sys.path.insert(0, str(project_root / "src"))
from wine_quality.data import FEATURE_COLUMNS, REQUIRED_COLUMNS, prepare_dataset

data_path = project_root / "data" / "raw" / "WineQT.csv"
frame = pd.read_csv(data_path)
print(f"Arquivo: {data_path}")
print(f"Dimensão bruta: {frame.shape[0]:,} linhas x {frame.shape[1]} colunas".replace(",", "."))

Arquivo: C:\Users\guira\OneDrive\SuperCerébro\03_Estudos_e_Recursos\Pos_FIAP\Tech-Challenge-Fase-2-Classificacao-de-Vinhos\data\raw\WineQT.csv
Dimensão bruta: 1.143 linhas x 13 colunas


## 1. Contrato estrutural do arquivo

In [2]:
expected_columns = set(REQUIRED_COLUMNS)
actual_columns = set(frame.columns)

checks = [
    {
        "validação": "Arquivo não vazio",
        "resultado": not frame.empty,
        "evidência": f"{len(frame):,} linhas".replace(",", "."),
    },
    {
        "validação": "Colunas obrigatórias presentes",
        "resultado": expected_columns.issubset(actual_columns),
        "evidência": f"ausentes: {sorted(expected_columns - actual_columns) or 'nenhuma'}",
    },
    {
        "validação": "Nenhuma coluna inesperada",
        "resultado": actual_columns == expected_columns,
        "evidência": f"extras: {sorted(actual_columns - expected_columns) or 'nenhuma'}",
    },
    {
        "validação": "Tipos numéricos",
        "resultado": all(
            pd.api.types.is_numeric_dtype(frame[column])
            for column in REQUIRED_COLUMNS
        ),
        "evidência": "features, quality e Id",
    },
]

contract_report = pd.DataFrame(checks)
display(contract_report)
assert contract_report["resultado"].all()

,validação,resultado,evidência
0,Arquivo não vazio,True,1.143 linhas
1,Colunas obrigatórias presentes,True,ausentes: nenhuma
2,Nenhuma coluna inesperada,True,extras: nenhuma
3,Tipos numéricos,True,"features, quality e Id"


In [3]:
profile = pd.DataFrame(
    {
        "tipo": frame.dtypes.astype(str),
        "nulos": frame.isna().sum(),
        "valores_únicos": frame.nunique(),
        "mínimo": frame.min(numeric_only=True),
        "máximo": frame.max(numeric_only=True),
    }
)
display(profile)

,tipo,nulos,valores_únicos,mínimo,máximo
fixed acidity,float64,0,91,4.60000,15.90000
volatile acidity,float64,0,135,0.12000,1.58000
citric acid,float64,0,77,0.00000,1.00000
residual sugar,float64,0,80,0.90000,15.50000
chlorides,float64,0,131,0.01200,0.61100
free sulfur dioxide,float64,0,53,1.00000,68.00000
total sulfur dioxide,float64,0,138,6.00000,289.00000
density,float64,0,388,0.99007,1.00369
pH,float64,0,87,2.74000,4.01000
sulphates,float64,0,89,0.33000,2.00000


## 2. Nulos, duplicidades e rastreabilidade

In [4]:
duplicate_rows = int(frame.duplicated().sum())
duplicate_ids = int(frame["Id"].duplicated().sum())
missing_values = int(frame.isna().sum().sum())

integrity_report = pd.DataFrame(
    [
        {
            "validação": "Valores ausentes",
            "quantidade": missing_values,
            "resultado": missing_values == 0,
            "decisão": "nenhuma imputação necessária na base bruta",
        },
        {
            "validação": "Linhas duplicadas",
            "quantidade": duplicate_rows,
            "resultado": duplicate_rows == 0,
            "decisão": "nenhuma linha removida",
        },
        {
            "validação": "Id duplicado",
            "quantidade": duplicate_ids,
            "resultado": duplicate_ids == 0,
            "decisão": "Id fica para rastreabilidade, não para predição",
        },
    ]
)
display(integrity_report)
assert integrity_report["resultado"].all()

,validação,quantidade,resultado,decisão
0,Valores ausentes,0,True,nenhuma imputação necessária na base bruta
1,Linhas duplicadas,0,True,nenhuma linha removida
2,Id duplicado,0,True,"Id fica para rastreabilidade, não para predição"


O `Id` identifica a amostra, mas não é uma medição físico-química. Por isso,
ele permanece no arquivo bruto para auditoria e é removido dos preditores.
A coluna `quality` também não entra como feature: ela é a origem do rótulo e
seu uso como entrada causaria *target leakage*.

## 3. Quais classificações são possíveis?

In [5]:
quality_distribution = (
    frame["quality"]
    .value_counts()
    .sort_index()
    .rename_axis("quality")
    .reset_index(name="amostras")
)
quality_distribution["participação"] = (
    quality_distribution["amostras"] / len(frame)
)
display(
    quality_distribution.style.format({"participação": "{:.1%}"})
)
print(
    "Classes originais observadas:",
    ", ".join(str(value) for value in quality_distribution["quality"]),
)

,quality,amostras,participação
0,3,6,0.5%
1,4,33,2.9%
2,5,483,42.3%
3,6,462,40.4%
4,7,143,12.5%
5,8,16,1.4%


Classes originais observadas: 3, 4, 5, 6, 7, 8


In [6]:
classification_options = pd.DataFrame(
    [
        {
            "opção": "Multiclasse original",
            "classes": "3, 4, 5, 6, 7 e 8",
            "interpretação": "prever a nota discreta de qualidade",
            "status": "possível, mas não é o recorte pedido",
        },
        {
            "opção": "Binária adotada",
            "classes": "0 = quality < 7; 1 = quality >= 7",
            "interpretação": "priorizar amostras de alta qualidade",
            "status": "recorte do enunciado",
        },
    ]
)
display(classification_options)

,opção,classes,interpretação,status
0,Multiclasse original,"3, 4, 5, 6, 7 e 8",prever a nota discreta de qualidade,"possível, mas não é o recorte pedido"
1,Binária adotada,0 = quality < 7; 1 = quality >= 7,priorizar amostras de alta qualidade,recorte do enunciado


Embora a base contenha seis notas distintas, o desafio orienta uma decisão
binária. O limiar adotado é `7`: notas 3 a 6 ficam em **baixa/média** e notas
7 e 8 ficam em **alta qualidade**.

In [7]:
features, target = prepare_dataset(frame, threshold=7)
binary_distribution = pd.DataFrame(
    {
        "classe": ["Baixa/média (< 7)", "Alta (>= 7)"],
        "rótulo": [0, 1],
        "amostras": [(target == 0).sum(), (target == 1).sum()],
    }
)
binary_distribution["participação"] = (
    binary_distribution["amostras"] / len(target)
)
display(
    binary_distribution.style.format({"participação": "{:.1%}"})
)
assert set(target.unique()) == {0, 1}

,classe,rótulo,amostras,participação
0,Baixa/média (< 7),0,984,86.1%
1,Alta (>= 7),1,159,13.9%


## 4. Saneamento aplicado e outliers sinalizados

In [8]:
outlier_rows = []
for column in FEATURE_COLUMNS:
    q1 = frame[column].quantile(0.25)
    q3 = frame[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((frame[column] < lower) | (frame[column] > upper)).sum())
    outlier_rows.append(
        {
            "variável": column,
            "limite_inferior": lower,
            "limite_superior": upper,
            "outliers_IQR": count,
            "participação": count / len(frame),
        }
    )

outlier_report = pd.DataFrame(outlier_rows).sort_values(
    "outliers_IQR", ascending=False
)
display(
    outlier_report.style.format(
        {
            "limite_inferior": "{:.3f}",
            "limite_superior": "{:.3f}",
            "participação": "{:.1%}",
        }
    )
)

,variável,limite_inferior,limite_superior,outliers_IQR,participação
3,residual sugar,0.850,3.650,110,9.6%
4,chlorides,0.040,0.120,77,6.7%
0,fixed acidity,4.100,12.100,44,3.8%
9,sulphates,0.280,1.000,43,3.8%
6,total sulfur dioxide,-39.000,121.000,40,3.5%
7,density,0.992,1.001,36,3.1%
8,pH,2.913,3.692,20,1.7%
5,free sulfur dioxide,-14.000,42.000,18,1.6%
1,volatile acidity,0.021,1.011,14,1.2%
10,alcohol,7.100,13.500,12,1.0%


In [9]:
sanitation_summary = pd.DataFrame(
    [
        {
            "item": "Nulos",
            "tratamento": "nenhuma linha alterada; não há nulos",
        },
        {
            "item": "Duplicidades",
            "tratamento": "nenhuma linha removida; Id é único",
        },
        {
            "item": "Id",
            "tratamento": "mantido para rastreabilidade e excluído de X",
        },
        {
            "item": "quality",
            "tratamento": "mantido como origem do alvo e excluído de X",
        },
        {
            "item": "Outliers",
            "tratamento": "sinalizados por IQR, sem remoção automática",
        },
        {
            "item": "Desbalanceamento",
            "tratamento": "mantido; split estratificado e pesos balanceados",
        },
    ]
)
display(sanitation_summary)
print(
    f"Matriz final de preditores: {features.shape[0]} linhas x "
    f"{features.shape[1]} features"
)

,item,tratamento
0,Nulos,nenhuma linha alterada; não há nulos
1,Duplicidades,nenhuma linha removida; Id é único
2,Id,mantido para rastreabilidade e excluído de X
3,quality,mantido como origem do alvo e excluído de X
4,Outliers,"sinalizados por IQR, sem remoção automática"
5,Desbalanceamento,mantido; split estratificado e pesos balanceados


Matriz final de preditores: 1143 linhas x 11 features


### Conclusão

A base está consistente para seguir à EDA: não há nulos, duplicidades ou
`Id` repetido; todas as colunas são numéricas; e o alvo binário preserva as
duas classes. Nenhum registro foi excluído. O principal cuidado para a
modelagem é o desbalanceamento: apenas 13,91% das amostras são de alta
qualidade.